## Imports

imports do projeto

In [ ]:
import os
from decimal import Decimal

import pandas as pd
from PIL import Image
from rembg import new_session, remove

from models.indice_linha import IndiceLinha

## Configuração

In [ ]:
# Diretórios
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
INDICE_PATH = os.path.join(DATA_DIR, 'Indice.xlsx')
GENERATED_DIR = os.path.join(BASE_DIR, 'generated')
ORIGINAL_PHOTOS_DIR = os.path.join(DATA_DIR, 'Original')
MASCARA_DIR = os.path.join(DATA_DIR, 'Mascara')
SEGMENTED_PHOTOS_DIR = os.path.join(GENERATED_DIR, 'Segmentada')

# Nomes das colunas do excel
NOME_COL = 'nome do arquivo'
FAZENDA_COL = 'fazenda'
PESO_COL = 'peso'

# Configuração do tipo de arquivo
ORIGINAL_IMAGE_TYPE = '.jpg'
REMBG_IMAGE_TYPE = '.jpg'

# Modelos para avaliação
MODELOS_PARA_AVALIACAO = [
    'u2net',
    'u2netp',
    'u2net_human_seg',
    # 'u2net_cloth_seg', este não vai entrar porque é específico para roupas e altera a resolução da imagem
    'silueta',
    'isnet-general-use',
    'isnet-anime',
    'sam',
    'birefnet-general',
    'birefnet-general-lite',
    'birefnet-portrait',
    'birefnet-dis',
    'birefnet-hrsod',
    'birefnet-cod',
    'birefnet-massive'
]

## Leitura do índice


In [ ]:
indice_df = pd.read_excel(INDICE_PATH)
colunas = {c.strip().lower(): c for c in indice_df.columns}

nome_col = colunas.get(NOME_COL)
fazenda_col = colunas.get(FAZENDA_COL)
peso_col = colunas.get(PESO_COL)

if not nome_col or not fazenda_col or not peso_col:
    raise ValueError('Alguma das colunas esperadas não foi encontrada no arquivo Excel.')

indice_excel = [
    IndiceLinha(
        nome_arquivo=str(row[nome_col]),
        fazenda=str(row[fazenda_col]),
        peso=Decimal(str(row[peso_col]))
    )
    for _, row in indice_df.iterrows()
]

## Processa as imagens

In [ ]:
# TODO: Quem sabe arrumar uma forma de fazer paralelização aqui?
for modelo in MODELOS_PARA_AVALIACAO:
    print(f"Avaliando modelo: {modelo}")

    # Cria uma sessão com o modelo
    rembg_session = new_session(modelo)

    # Cria o diretório de saída, se não existir
    output_dir = os.path.join(SEGMENTED_PHOTOS_DIR, modelo)
    os.makedirs(output_dir, exist_ok=True)

    for linha in indice_excel:
        # Armazena em variáveis e verifica se os arquivos existem
        original_path = os.path.join(ORIGINAL_PHOTOS_DIR, f"{linha.nome_arquivo}{ORIGINAL_IMAGE_TYPE}")
        if not os.path.isfile(original_path):
            raise FileNotFoundError(f"   ! Arquivo original não encontrado: {original_path}")

        mascara_path = os.path.join(MASCARA_DIR, f"{linha.nome_arquivo}{ORIGINAL_IMAGE_TYPE}")
        if not os.path.isfile(mascara_path):
            raise FileNotFoundError(f"   ! Arquivo de máscara não encontrado: {mascara_path}")

        # Verifica se a imagem já foi processada e se está íntegra (não corrompida)
        output_path = os.path.join(SEGMENTED_PHOTOS_DIR, modelo, f"{linha.nome_arquivo}{REMBG_IMAGE_TYPE}")
        skip_processing = False
        if os.path.isfile(output_path):
            try:
                # Tenta abrir e verificar a imagem
                with Image.open(output_path) as img:
                    img.verify()  # Verifica se a imagem não está corrompida
                # Reabre para verificar se pode carregar os dados
                with Image.open(output_path) as img:
                    img.load()  # Força o carregamento dos dados da imagem
                print(f" - Pulando foto já processada e íntegra: {linha.nome_arquivo}")
                skip_processing = True
            except (IOError, SyntaxError, Exception) as e:
                print(f" - Arquivo corrompido detectado, reprocessando: {linha.nome_arquivo}")
                print(f"   - Erro: {e}")
                skip_processing = False
        
        if skip_processing:
            continue

        # Processa a imagem
        input_rembg = Image.open(original_path)
        output_rembg = remove(input_rembg, session=rembg_session)

        # Salva a imagem processada (converte RGBA para RGB com fundo branco)
        if output_rembg.mode == 'RGBA':
            background = Image.new('RGB', output_rembg.size, (255, 255, 255))
            background.paste(output_rembg, mask=output_rembg.split()[3])
            background.save(output_path, format="JPEG", quality=95)
        else:
            output_rembg.save(output_path, format="JPEG", quality=95)

        # Prints de debug
        print(f" - Processando foto: {linha.nome_arquivo}")
        print(f"   - Original: {original_path}")
        print(f"   - Mascara: {mascara_path}")
        print(f"   - Destino: {output_path}")

